# Bayesian KAN — Interpretability with Uncertainty

Demonstrates the KAN interpretability workflow (Liu et al. 2024, ICLR) extended
to the Bayesian setting: each learned activation function is a **distribution
over functions**, not a single curve.

**Method.** For interpretability we train in **pure-spline mode** (`base_fun=None`):
the residual SiLU branch is disabled so the B-spline alone carries the signal.
With the residual active, every edge is dominated by SiLU and symbolic fits
collapse onto that single shape. The grid range is matched to the data range so
boundary spline coefficients remain data-constrained.

**Symbolic regression** uses the four-parameter affine fit `y ~= c*f(a*x+b)+d`
(Liu et al. 2024; pykan `fit_params`), with a complexity tie-breaker so the
simpler form is chosen when several candidates fit comparably well.

**Bayesian extension.** Each activation has a posterior mean E[phi(x)] and
standard deviation Std[phi(x)], derived analytically from the B-spline
coefficient posterior (`BayesianKANLayer.get_activation_stats`). We report the
symbolic fit R^2 on the mean and on the +/-2 sigma bounds.

**Test functions** (Liu et al. 2024, Fig. 3):
1. `f(x) = x^2`           — single edge, should recover `x^2`
2. `f(x) = sin(pi x)`     — single edge, should recover `sin` with a~=pi
3. `f(x,y) = x^2+sin(pi y)` — additive [2->1], decomposes to `x^2` and `sin`
4. `f(x1,x2) = exp(sin(pi(x1^2+x2^2)))` — Liu et al. composition (hard case)


## 1. Setup

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt

from bkan.models import BayesianKAN, suggest_symbolic, format_symbolic_table
from bkan.training.trainer import BNNTrainer, TrainingConfig

torch.manual_seed(0)
np.random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


## 2. Helpers

In [ ]:
def make_data_1d(fn, n=1000, x_range=(-1, 1), noise=0.02, seed=0):
    rng = np.random.RandomState(seed)
    x = rng.uniform(*x_range, size=n)
    y = fn(x) + noise * rng.randn(n)
    return (torch.FloatTensor(x.reshape(-1, 1)),
            torch.FloatTensor(y.reshape(-1, 1)))


def make_data_2d(fn, n=3000, x_range=(-1, 1), noise=0.02, seed=0):
    rng = np.random.RandomState(seed)
    X = rng.uniform(*x_range, size=(n, 2))
    y = fn(X[:, 0], X[:, 1]) + noise * rng.randn(n)
    return torch.FloatTensor(X), torch.FloatTensor(y.reshape(-1, 1))


def train_interp_bkan(x, y, input_dim, hidden_dims, x_range,
                      epochs=3000, lr=1e-2, num=10, seed=0):
    """Train a pure-spline BKAN for symbolic interpretability.

    Pure spline (base_fun=None) and grid matched to the data range are the
    key settings for faithful symbolic recovery."""
    torch.manual_seed(seed)
    model = BayesianKAN(
        input_dim=input_dim, hidden_dims=hidden_dims, output_dim=1,
        num=num, k=3, prior_std=1.0, learn_noise=False,
        coef_log_var_init=-5.0, grid_range=list(x_range),
        base_fun=None, device=DEVICE,
    )
    cfg = TrainingConfig(
        epochs=epochs, batch_size=1000, learning_rate=lr,
        kl_annealing_epochs=200, n_mc_samples=1,
        early_stopping_patience=epochs, scheduler='cosine',
        verbose=True, print_every=max(epochs // 3, 1),
    )
    BNNTrainer(model, cfg, DEVICE).train(x, y)
    with torch.no_grad():
        pred, _ = model(x.to(DEVICE), sample=False)
    rmse = torch.sqrt(((pred.cpu() - y) ** 2).mean()).item()
    print(f'  fit RMSE = {rmse:.4f}')
    return model


def plot_activation(ax, layer, in_idx, out_idx, x_range, color='C0',
                    n_grid=300, true_fn=None):
    """Plot phi_{in,out}(x) with posterior +/-2 sigma band (Liu 2024 style)."""
    x_grid = torch.linspace(x_range[0], x_range[1], n_grid)
    mean_phi, std_phi = layer.get_activation_stats(in_idx, out_idx, x_grid)
    x_np, m_np, s_np = x_grid.numpy(), mean_phi.numpy(), std_phi.numpy()

    ax.fill_between(x_np, m_np - 2*s_np, m_np + 2*s_np, alpha=0.25,
                    color=color, label=r'$\pm 2\sigma$')
    ax.plot(x_np, m_np, color=color, lw=2.0, label='posterior mean')
    if true_fn is not None:
        yt = true_fn(x_np)
        A = np.column_stack([yt, np.ones_like(yt)])
        ab, *_ = np.linalg.lstsq(A, m_np, rcond=None)
        ax.plot(x_np, ab[0]*yt + ab[1], 'k--', lw=1.1, alpha=0.6,
                label='true (scaled)')
    ax.set_xlabel('x'); ax.set_ylabel(r'$\varphi(x)$')
    ax.grid(True, alpha=0.25); ax.legend(fontsize=7)
    return x_np, m_np, s_np


print('Helpers defined.')


## 3. Case 1: f(x) = x^2

Single-edge [1->1] KAN. The one activation should recover `x^2`.

In [ ]:
fn_sq = lambda x: x**2
x1, y1 = make_data_1d(fn_sq, x_range=(-1, 1))
model1 = train_interp_bkan(x1, y1, input_dim=1, hidden_dims=[], x_range=(-1, 1))


In [ ]:
xg = np.linspace(-1, 1, 300); xgt = torch.FloatTensor(xg)
layer = model1.output_layer
m_np, s_np = [t.numpy() for t in layer.get_activation_stats(0, 0, xgt)]
res = suggest_symbolic(xg, m_np, s_np, n_top=5)

fig, ax = plt.subplots(figsize=(5.5, 4))
plot_activation(ax, layer, 0, 0, (-1, 1), color='C0', true_fn=fn_sq)
best = res[0]
ax.set_title(f'f(x)=x^2  ->  best: {best["name"]}  '
             f'R2={best["r2_mean"]:.4f}', fontsize=10)
plt.tight_layout(); plt.savefig('interp_case1_xsq.pdf', bbox_inches='tight'); plt.show()

print('Symbolic regression (edge 0->0):')
print(format_symbolic_table(res))


## 4. Case 2: f(x) = sin(pi x)

Single-edge [1->1] KAN. Should recover `sin` with input scale a ~= pi. The
four-parameter fit is essential here: without the input scale, a fixed `sin(x)`
cannot match the frequency of `sin(pi x)`.

In [ ]:
fn_sin = lambda x: np.sin(np.pi * x)
x2, y2 = make_data_1d(fn_sin, x_range=(-1, 1))
model2 = train_interp_bkan(x2, y2, input_dim=1, hidden_dims=[], x_range=(-1, 1), lr=5e-2)


In [ ]:
layer = model2.output_layer
m_np, s_np = [t.numpy() for t in layer.get_activation_stats(0, 0, xgt)]
res = suggest_symbolic(xg, m_np, s_np, n_top=5)

fig, ax = plt.subplots(figsize=(5.5, 4))
plot_activation(ax, layer, 0, 0, (-1, 1), color='C2', true_fn=fn_sin)
best = res[0]
ax.set_title(f'f(x)=sin(pi x)  ->  best: {best["name"]}  '
             f'R2={best["r2_mean"]:.4f}  (a={best["a"]:.3f})', fontsize=10)
plt.tight_layout(); plt.savefig('interp_case2_sin.pdf', bbox_inches='tight'); plt.show()

print('Symbolic regression (edge 0->0):')
print(format_symbolic_table(res))
print(f'\nRecovered input scale a = {best["a"]:.4f}  (true frequency = pi = {np.pi:.4f})')


## 5. Case 3: f(x,y) = x^2 + sin(pi y)

Additive function on an [2->1] KAN (two edges summed at the output node). The
edge from x should recover `x^2`; the edge from y should recover `sin`. This is
the natural Kolmogorov-Arnold decomposition of an additive function.

In [ ]:
fn_add = lambda x, y: x**2 + np.sin(np.pi * y)
X3, y3 = make_data_2d(fn_add, x_range=(-1, 1))
model3 = train_interp_bkan(X3, y3, input_dim=2, hidden_dims=[], x_range=(-1, 1),
                           epochs=5000, lr=5e-2)


In [ ]:
layer = model3.output_layer
true_fns = {0: lambda x: x**2, 1: lambda x: np.sin(np.pi * x)}
labels = ['x', 'y']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i in range(2):
    m_np, s_np = [t.numpy() for t in layer.get_activation_stats(i, 0, xgt)]
    res = suggest_symbolic(xg, m_np, s_np, n_top=5)
    plot_activation(axes[i], layer, i, 0, (-1, 1), color='C3', true_fn=true_fns[i])
    best = res[0]
    axes[i].set_title(f'edge ({labels[i]} -> out)  best: {best["name"]}  '
                      f'R2={best["r2_mean"]:.4f}', fontsize=10)
    print(f'Edge ({labels[i]} -> output):')
    print(format_symbolic_table(res))
    print()

fig.suptitle(r'$f(x,y)=x^2+\sin(\pi y)$  — additive [2->1] decomposition', fontsize=11)
plt.tight_layout(); plt.savefig('interp_case3_additive.pdf', bbox_inches='tight'); plt.show()


## 6. Case 4: f(x1,x2) = exp(sin(pi(x1^2 + x2^2)))

Liu et al. (2024) Fig. 3 example. This is a genuine **composition** (not additively
separable), so individual edges do not map to single clean library functions:
the first layer must produce x^2-like terms, which are summed and passed through
sin then exp. We report the per-edge fits honestly — high R^2 on inner-layer
x^2 edges, lower/ambiguous fits where the composition is distributed. A [2,2,1]
architecture is used following the paper.

In [ ]:
fn_liu = lambda x, y: np.exp(np.sin(np.pi * (x**2 + y**2)))
X4, y4 = make_data_2d(fn_liu, n=3000, x_range=(-1, 1), noise=0.01)
model4 = train_interp_bkan(X4, y4, input_dim=2, hidden_dims=[2], x_range=(-1, 1),
                           epochs=8000, lr=5e-2)


In [ ]:
L0 = model4.hidden_layers[0]   # [2 -> 2]
L1 = model4.output_layer        # [2 -> 1]
in_lbl = ['x1', 'x2']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
# Layer 0: 2x2 edges
for i in range(2):
    for j in range(2):
        ax = axes[0, i*2 + j] if (i*2 + j) < 3 else axes[1, (i*2 + j) - 3]
        m_np, s_np = [t.numpy() for t in L0.get_activation_stats(i, j, xgt)]
        res = suggest_symbolic(xg, m_np, s_np, n_top=1)
        plot_activation(ax, L0, i, j, (-1, 1), color='C4')
        ax.set_title(f'L0 ({in_lbl[i]}->h{j})  {res[0]["name"]}  '
                     f'R2={res[0]["r2_mean"]:.3f}', fontsize=8)
# Layer 1: 2 edges
for i in range(2):
    ax = axes[1, i + 1]
    m_np, s_np = [t.numpy() for t in L1.get_activation_stats(i, 0, xgt)]
    res = suggest_symbolic(xg, m_np, s_np, n_top=1)
    plot_activation(ax, L1, i, 0, (-1, 1), color='C5')
    ax.set_title(f'L1 (h{i}->out)  {res[0]["name"]}  '
                 f'R2={res[0]["r2_mean"]:.3f}', fontsize=8)
axes[1, 0].set_visible(False)

fig.suptitle(r'$f=\exp(\sin(\pi(x_1^2+x_2^2)))$  — all edges, $\pm2\sigma$', fontsize=11)
plt.tight_layout(); plt.savefig('interp_case4_liu.pdf', bbox_inches='tight'); plt.show()


## 7. Summary

Symbolic recovery on the clean cases (1-3):

In [ ]:
def best_edge(layer, i, j):
    m, s = layer.get_activation_stats(i, j, xgt)
    r = suggest_symbolic(xg, m.numpy(), s.numpy(), n_top=1)[0]
    dr = (r['r2_upper'] - r['r2_lower']) if r['r2_upper'] is not None else float('nan')
    return r['name'], r['r2_mean'], dr

rows = [
    ('f(x)=x^2            edge(0->0)', 'x^2',    *best_edge(model1.output_layer, 0, 0)),
    ('f(x)=sin(pi x)      edge(0->0)', 'sin(x)', *best_edge(model2.output_layer, 0, 0)),
    ('f(x,y)=x^2+sin(piy) edge(x->o)', 'x^2',    *best_edge(model3.output_layer, 0, 0)),
    ('f(x,y)=x^2+sin(piy) edge(y->o)', 'sin(x)', *best_edge(model3.output_layer, 1, 0)),
]
hdr = f"{'Case / edge':>34}  {'expected':>9}  {'recovered':>10}  {'R2':>7}  {'dR2':>7}"
print(hdr); print('-' * len(hdr))
for name, exp, got, r2, dr in rows:
    flag = 'OK' if exp == got else '? '
    print(f'{name:>34}  {exp:>9}  {got:>10}  {r2:>7.4f}  {dr:>7.4f}  {flag}')
